# Clinical Digital-Twin Foundation: Reproducible Machine-Learning Analysis

**Study:** Development and validation of digital twin models for personalised simulation of diabetic and hypertensive medication efficacy  
**Researcher:** Nana Efua Bo Alan-Mensah  

This notebook provides a presentation-ready, reproducible proof-of-concept analysis of the available hospital dataset. It does **not** claim that the current cross-sectional dataset is sufficient to validate a longitudinal medication-response digital twin. Instead, it establishes a machine-learning baseline for identifying uncontrolled blood pressure, which is one component that can later support patient-specific simulation.

> **Presentation cue:** Explain that a clinical digital twin needs (1) a patient data representation, (2) a predictive or physiological model, (3) repeated updating with new observations, and (4) prospective validation. This notebook addresses the first two at proof-of-concept level.


## 1. Research question and modelling target

**Question:** Can routinely collected demographic and clinical context distinguish patients whose latest blood-pressure measurement is uncontrolled?

The binary outcome is defined as:

- **Uncontrolled:** systolic blood pressure ≥140 mmHg **or** diastolic blood pressure ≥90 mmHg.
- **Controlled:** both measurements are below those thresholds.

Blood-pressure measurements are used only to construct the outcome and are excluded from predictors to prevent target leakage. The threshold is an analytical definition for this proof of concept, not an individual treatment recommendation.

> **Presentation cue:** Target leakage occurs when the model is given information that directly reveals the answer. Removing systolic and diastolic pressure from the predictors makes the evaluation meaningful.


In [1]:
from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             RocCurveDisplay, PrecisionRecallDisplay)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATA_PATH = Path('hospitalData.xlsx')
assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH.resolve()}'
print('Environment ready. Dataset:', DATA_PATH.resolve())


Environment ready. Dataset: C:\Users\yamoa\OneDrive\Documents\MPhil\RA_\Thesis\MadamEfua\DigitalTwin\hospitalData.xlsx


## 2. Load data and protect privacy

The source workbook contains direct and quasi-identifiers. They are removed immediately after loading and are never used as model features. The original workbook is not modified.

> **Ethics cue:** De-identification is necessary but not sufficient. Formal ethical approval, access controls, purpose limitation and secure storage remain necessary for clinical research.


In [2]:
raw = pd.read_excel(DATA_PATH)
print(f'Raw records: {raw.shape[0]:,}; variables: {raw.shape[1]}')

identifier_columns = ['First Name', 'Last Name', 'Landmark']
required = ['Program ID', 'Latest Systolic', 'Latest Diastolic']
missing_required = [c for c in required if c not in raw.columns]
assert not missing_required, f'Required columns missing: {missing_required}'

# Program ID is retained only temporarily to consolidate repeated records.
analysis = raw.drop(columns=[c for c in identifier_columns if c in raw.columns]).copy()
print('Direct names and location field removed from the analysis frame.')


Raw records: 441; variables: 30
Direct names and location field removed from the analysis frame.


## 3. Data-quality assessment

We inspect completeness, repeated program IDs and plausible ranges before modelling. This stage is essential because an accurate algorithm cannot repair an ambiguous or poorly documented dataset.

> **Presentation cue:** Distinguish 441 *records* from 378 *people*. A person-level split prevents the same patient appearing in both training and evaluation data.


In [3]:
quality = pd.DataFrame({
    'dtype': analysis.dtypes.astype(str),
    'non_missing': analysis.notna().sum(),
    'missing_n': analysis.isna().sum(),
    'missing_pct': (analysis.isna().mean()*100).round(1),
    'unique_n': analysis.nunique(dropna=True)
}).sort_values('missing_pct', ascending=False)

print('Unique program IDs:', analysis['Program ID'].nunique())
print('Repeated records:', len(analysis)-analysis['Program ID'].nunique())
display(quality.head(12))


Unique program IDs: 378
Repeated records: 63
                               dtype  non_missing  missing_n  missing_pct  unique_n
Tobacco Status               float64            1        440         99.8         1
Alcohol Status               float64            5        436         98.9         5
Complication Status           object           42        399         90.5         7
Diabetes Known Patient Year  float64           82        359         81.4        22
Comorbidity                   object          127        314         71.2        64
Diabetes Patient Type         object          141        300         68.0         2
HTN Known Patient Year       float64          193        248         56.2        31
Tobacco Use                   object          298        143         32.4         2
Alcohol Use                   object          298        143         32.4         3
Physical Activity             object          310        131         29.7         4
Latest Glucose Type           o

## 4. Cohort construction

The study focuses on adults with hypertension, with or without Type 2 diabetes. Type 1 diabetes, pre-hypertension and unrelated diagnoses are excluded from this proof-of-concept cohort. Diagnosis labels are normalised because the same combined diagnosis appears in different orders.

For repeated program IDs, the record with the most recent available clinical date is retained. This produces one row per person and avoids optimistic validation.


In [4]:
def parse_date_series(s):
    return pd.to_datetime(s, errors='coerce', dayfirst=True)

date_cols = ['Date Of Last BP Assessment','Date Of Last BG Assessment',
             'Date Of Last Medical Review','Date Of Enrollment','Latest BMI Date']
for c in date_cols:
    if c in analysis.columns:
        analysis[c] = parse_date_series(analysis[c])

analysis['_latest_date'] = analysis[[c for c in date_cols if c in analysis]].max(axis=1)
analysis = analysis.sort_values('_latest_date').drop_duplicates('Program ID', keep='last')

dx = analysis['Confirmed Diagnosis'].fillna('').str.lower()
eligible = dx.str.contains('hypertension') & ~dx.str.contains('pre-hypertension') & ~dx.str.contains('type 1')
cohort = analysis.loc[eligible].copy()
cohort['Clinical Group'] = np.where(
    cohort['Confirmed Diagnosis'].str.contains('Type 2', case=False, na=False),
    'T2DM + hypertension', 'Hypertension only')

cohort['uncontrolled_bp'] = ((pd.to_numeric(cohort['Latest Systolic'], errors='coerce') >= 140) |
                             (pd.to_numeric(cohort['Latest Diastolic'], errors='coerce') >= 90)).astype(int)
cohort = cohort.dropna(subset=['Latest Systolic','Latest Diastolic'])

print('Eligible unique patients:', len(cohort))
display(pd.crosstab(cohort['Clinical Group'], cohort['uncontrolled_bp'], margins=True)
        .rename(columns={0:'Controlled',1:'Uncontrolled'}))


Eligible unique patients: 326
uncontrolled_bp      Controlled  Uncontrolled  All
Clinical Group                                    
Hypertension only           134           103  237
T2DM + hypertension          42            47   89
All                         176           150  326


## 5. Feature engineering

Predictors are limited to information that can plausibly be available independently of the latest blood-pressure outcome:

- age, sex and BMI;
- diabetes/hypertension clinical group;
- glucose measurement and type;
- cardiovascular risk category;
- lifestyle fields;
- medication text converted into broad drug-class indicators.

Medication names are highly inconsistent in the source, so transparent class indicators are preferable to treating every spelling variant as a separate category.


In [5]:
def medication_flags(text):
    t = str(text).lower()
    groups = {
        'med_ccb': r'amlodipine|nifedipine',
        'med_raas': r'losartan|lisinopril|enalapril|valsartan|captopril',
        'med_diuretic': r'hydrochlor|bendro|furosemide|spironolactone',
        'med_beta_blocker': r'atenolol|bisoprolol|propranolol|carvedilol',
        'med_metformin': r'metformin',
        'med_sulfonylurea': r'gliclazide|glibenclamide|glimepiride',
        'med_insulin': r'insulin'
    }
    return {k:int(bool(re.search(p,t))) for k,p in groups.items()}

med = cohort['Medication Prescribed'].fillna('').apply(medication_flags).apply(pd.Series)
model_data = pd.concat([cohort.reset_index(drop=True), med.reset_index(drop=True)], axis=1)

numeric_features = ['Latest Age','Latest BMI','Latest Glucose Value'] + list(med.columns)
categorical_features = ['Gender','Clinical Group','Latest Glucose Type',
                        'Latest CVD Risk Score','Physical Activity','Alcohol Use','Tobacco Use']
X = model_data[numeric_features + categorical_features].copy()
y = model_data['uncontrolled_bp'].astype(int)

print('Modelling sample:', len(X))
print('Outcome prevalence:', f'{y.mean():.1%} uncontrolled')
display(X.head(3).assign(outcome=y.head(3).values))


Modelling sample: 326
Outcome prevalence: 46.0% uncontrolled
   Latest Age  Latest BMI  Latest Glucose Value  med_ccb  med_raas  med_diuretic  med_beta_blocker  med_metformin  med_sulfonylurea  med_insulin  Gender       Clinical Group Latest Glucose Type   Latest CVD Risk Score Physical Activity Alcohol Use Tobacco Use  outcome
0          79       22.22                   NaN        1         0             1                 0              0                 0            0    male    Hypertension only                 NaN       Medium risk - 13%               NaN         NaN         NaN        0
1          49       59.52                   NaN        0         0             0                 0              0                 0            0  female    Hypertension only                 NaN           Low risk - 3%               NaN         NaN         NaN        1
2          87       49.30                  12.0        0         0             0                 0              0                 0 

## 6. Reproducible modelling pipeline

Two models are compared:

1. **Logistic regression** – interpretable baseline.
2. **Random forest** – nonlinear comparison model.

Missing numeric values are median-imputed; missing categorical values use the most frequent category. Categorical fields are one-hot encoded. All transformations occur inside each cross-validation fold, preventing information leakage.

Five-fold stratified cross-validation is used. Reported uncertainty is the standard deviation across folds, not a clinical confidence interval.


In [6]:
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocess = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

models = {
    'Logistic regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random forest': RandomForestClassifier(n_estimators=400, min_samples_leaf=4,
                                             class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'accuracy':'accuracy','precision':'precision','recall':'recall','f1':'f1','roc_auc':'roc_auc'}

rows=[]
for name, estimator in models.items():
    pipe=Pipeline([('preprocess',preprocess),('model',estimator)])
    scores=cross_validate(pipe,X,y,cv=cv,scoring=scoring,n_jobs=-1)
    for metric in scoring:
        vals=scores['test_'+metric]
        rows.append({'Model':name,'Metric':metric,'Mean':vals.mean(),'SD':vals.std(ddof=1)})
results=pd.DataFrame(rows)
display(results.pivot(index='Metric',columns='Model',values='Mean').round(3))


Model      Logistic regression  Random forest
Metric                                       
accuracy                 0.635          0.653
f1                       0.606          0.624
precision                0.608          0.631
recall                   0.607          0.627
roc_auc                  0.688          0.692


## 7. Evaluation plots and error analysis

ROC-AUC measures ranking across thresholds; F1 balances precision and recall. Because screening applications may value sensitivity, recall should be examined alongside overall accuracy.

> **Presentation cue:** A useful model must be compared with a simple baseline and evaluated on unseen patients. Performance alone does not establish clinical utility.


In [7]:
best_name = results.loc[results.query("Metric == 'roc_auc'")['Mean'].idxmax(), 'Model']
best_pipe = Pipeline([('preprocess',preprocess),('model',models[best_name])])
pred = cross_val_predict(best_pipe,X,y,cv=cv,method='predict',n_jobs=-1)
prob = cross_val_predict(best_pipe,X,y,cv=cv,method='predict_proba',n_jobs=-1)[:,1]

print('Selected model:',best_name)
print(classification_report(y,pred,target_names=['Controlled','Uncontrolled']))
print('Cross-validated ROC-AUC:',round(roc_auc_score(y,prob),3))

fig,axes=plt.subplots(1,3,figsize=(15,4))
sns.heatmap(confusion_matrix(y,pred),annot=True,fmt='d',cmap='Blues',cbar=False,ax=axes[0])
axes[0].set(title='Cross-validated confusion matrix',xlabel='Predicted',ylabel='Observed')
RocCurveDisplay.from_predictions(y,prob,ax=axes[1],name=best_name)
PrecisionRecallDisplay.from_predictions(y,prob,ax=axes[2],name=best_name)
plt.tight_layout(); plt.show()


Selected model: Random forest
              precision    recall  f1-score   support

  Controlled       0.68      0.68      0.68       176
Uncontrolled       0.62      0.63      0.62       150

    accuracy                           0.65       326
   macro avg       0.65      0.65      0.65       326
weighted avg       0.65      0.65      0.65       326

Cross-validated ROC-AUC: 0.689


## 8. Interpretation and limitations

This analysis provides a defensible machine-learning baseline, not a completed clinical digital twin.

### What the analysis demonstrates

- A privacy-conscious patient-level analysis workflow.
- Transparent cohort and outcome definitions.
- Reproducible preprocessing and cross-validation.
- A baseline for later comparison with physiological and longitudinal models.

### What it cannot demonstrate

- Causal medication efficacy: treatment was not randomly assigned.
- Longitudinal response: most fields describe only the latest assessment.
- Dose-response relationships: medication dose, adherence and treatment timing are incomplete.
- External validity: all records come from one source and no external test cohort is supplied.
- Deployment readiness: thresholds, calibration, fairness and clinical workflow impact require prospective evaluation.

> **Presentation cue:** Describe the result as a *digital-twin foundation* or *proof of concept*. The next scientific step is to acquire repeated pre/post-medication measurements, dosage, adherence and laboratory data.


In [8]:
summary = results.copy()
summary['Mean_SD'] = summary.apply(lambda r: f"{r['Mean']:.3f} ± {r['SD']:.3f}", axis=1)
summary[['Model','Metric','Mean','SD','Mean_SD']].to_csv('model_cv_metrics.csv',index=False)

metadata = {
    'source_records': int(len(raw)),
    'unique_program_ids': int(raw['Program ID'].nunique()),
    'eligible_unique_patients': int(len(cohort)),
    'uncontrolled_n': int(y.sum()),
    'uncontrolled_pct': float(y.mean()*100),
    'selected_model': best_name,
    'cross_validated_roc_auc': float(roc_auc_score(y,prob))
}
Path('analysis_summary.json').write_text(json.dumps(metadata,indent=2))
print(json.dumps(metadata,indent=2))
print('Saved model_cv_metrics.csv and analysis_summary.json')


{
  "source_records": 441,
  "unique_program_ids": 378,
  "eligible_unique_patients": 326,
  "uncontrolled_n": 150,
  "uncontrolled_pct": 46.012269938650306,
  "selected_model": "Random forest",
  "cross_validated_roc_auc": 0.688939393939394
}
Saved model_cv_metrics.csv and analysis_summary.json


## 9. Roadmap from baseline model to a validated digital twin

1. **Data governance:** confirm ethics approval, consent/waiver, data controller permissions and secure pseudonymisation.
2. **Longitudinal dataset:** collect dates, doses, adherence, pre/post biomarkers and medication changes.
3. **Physiological layer:** calibrate glucose-insulin and cardiovascular models using repeated observations.
4. **Learning layer:** estimate patient-specific parameters and quantify uncertainty.
5. **Validation:** use temporal and external validation, calibration, subgroup fairness and clinical decision-curve analysis.
6. **Prospective study:** test whether twin-supported decisions improve outcomes without unacceptable harm.

### Suggested concluding sentence for the presentation

“The present work establishes a reproducible clinical-data and machine-learning foundation; longitudinal and prospective validation are the necessary next steps before the system can be considered a clinical digital twin.”
